# CR Behavior Anomaly Detection — LLM Matching

This notebook checks whether each CR traded goods match
its registered business activities (ISIC4) using LLM reasoning.

**Logic:**
- Each row has one active CR
- Each CR has one or more ISIC4 registered activities in the sijilat file
- The hs_desc column describes what the company actually traded
- The LLM decides whether the goods match the registered activities
- A mismatch is flagged with a risk level and a reason


## Step 1 — Imports and Paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time
from openai import OpenAI

# Parameters (injected by pipeline_runner)
input_path = r"C:\Users\fawaz\Desktop\Project\outputs\dummy_2020_cleaned.csv"
year       = "2020"
output_path = r"C:\Users\fawaz\Desktop\Project\outputs\CR_LLM.csv"

In [2]:
# Paths
CLEANED_PATH = Path(input_path)
SIJILAT_PATH = Path(r"C:\Users\fawaz\Desktop\mofne\Ecrypted dataset\sijilat_company_business_activities (1).xlsx")
OUTPUT_ALL   = Path(output_path)

# API KEY LOADING from .env
import os
from dotenv import load_dotenv

env_path = Path(r"C:\Users\fawaz\Desktop\Project\.env")
load_dotenv(dotenv_path=env_path, override=True)

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

# Sanity check — fail loudly if either key is missing
if not OPENAI_API_KEY:
    raise RuntimeError(f"OPENAI_API_KEY not found in {env_path}")
if not DEEPSEEK_API_KEY:
    raise RuntimeError(f"DEEPSEEK_API_KEY not found in {env_path}")

print(f"Loaded keys from {env_path}")

Loaded keys from C:\Users\fawaz\Desktop\Project\.env


## Step 2 — Load Cleaned Trade Data

In [3]:
df = pd.read_csv(CLEANED_PATH, low_memory=False, parse_dates=["declaration_date"])
print(f"Cleaned data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df[["hs_clean", "hs_desc", "active_cr", "trade_type"]].head(3)

Cleaned data shape: (1000, 52)
Columns: ['item_id', 'declaration_id', 'year', 'customs_office_code', 'customs_office_name', 'regime', 'registration_serial', 'registration_number', 'reference_number', 'registration_date', 'hs_code', 'commercial_description', 'country_of_origin', 'country_of_origin_code', 'country_of_export', 'country_of_export_code', 'country_of_destination', 'country_of_destination_code', 'uom', 'qty_by_uom', 'actual_unit_price', 'price_basis', 'local_amount', 'sup_amount', 'invoice_amount', 'net_weight', 'gross_weight', 'status', 'specification_code', 'warehouse_code', 'exit_office_code', 'exit_officer_id', 'operation_name', 'operation_date', 'encrypted_declarant_cr', 'encrypted_consignee_cr', 'encrypted_exporter_cr', 'declaration_date', 'year_month', 'hs_clean', 'hs2', 'hs6', 'is_reexport', 'trade_type', 'local_amount_clean', 'invoice_amount_clean', 'sup_amount_clean', 'net_weight_clean', 'gross_weight_clean', 'partner_country_code', 'active_cr', 'hs_desc']


,hs_clean,hs_desc,active_cr,trade_type
0,847130000000,Portable laptop computers,CR_HASH_036,Import
1,870322100000,Passenger cars 1000-1500cc,CR_HASH_031,Re-export
2,392330100000,Plastic bottles for packaging,CR_HASH_016,Re-export


## Step 3 — Load Sijilat

In [4]:
sijilat = pd.read_excel(SIJILAT_PATH)
sijilat = sijilat.drop(columns=["isic4_desc_ar", "isic4_sector_ar"], errors="ignore")
print(f"Sijilat shape: {sijilat.shape}")
print(f"Columns: {sijilat.columns.tolist()}")
sijilat.head(3)

Sijilat shape: (489993, 5)
Columns: ['isic4_code', 'isic4_desc_en', 'encrypted_cr', 'isic4_sector_id', 'isic4_sector_en']


,isic4_code,isic4_desc_en,encrypted_cr,isic4_sector_id,isic4_sector_en
0,469-1,General Trade,hzOkRPI8dbxbnkvJWJZRKg==,7.0,Sale/Trading Activities; repair of motor vehic...
1,4721-1,Sale/Trade of Food and Beverages,hzOkRPI8dbxbnkvJWJZRKg==,7.0,Sale/Trading Activities; repair of motor vehic...
2,4791,Retail sale via Internet,vXhzlCG8tYcCC5n6pWGPHg==,7.0,Sale/Trading Activities; repair of motor vehic...


## Step 4 — Join Sijilat onto Main Dataset

In [5]:
# Each CR may have multiple registered ISIC4 activities.
# We aggregate them into combined strings so the LLM sees the full picture per CR.

sijilat_grouped = (
    sijilat
    .groupby("encrypted_cr")
    .agg(
        isic4_codes   = ("isic4_code",      lambda x: ", ".join(x.dropna().unique())),
        isic4_descs   = ("isic4_desc_en",   lambda x: " | ".join(x.dropna().unique())),
        isic4_sectors = ("isic4_sector_en", lambda x: " | ".join(x.dropna().unique())),
    )
    .reset_index()
    .rename(columns={"encrypted_cr": "active_cr"})
)

df = df.merge(sijilat_grouped, on="active_cr", how="left")

print(f"Rows with ISIC4 info:  {df['isic4_descs'].notna().sum()}")
print(f"Rows missing ISIC4:    {df['isic4_descs'].isna().sum()}")
df[["active_cr", "hs_desc", "isic4_descs", "isic4_sectors"]].head(3)

Rows with ISIC4 info:  0
Rows missing ISIC4:    1000


,active_cr,hs_desc,isic4_descs,isic4_sectors
0,CR_HASH_036,Portable laptop computers,NaN,NaN
1,CR_HASH_031,Passenger cars 1000-1500cc,NaN,NaN
2,CR_HASH_016,Plastic bottles for packaging,NaN,NaN


## Step 5 — Build the LLM Prompt Field

In [6]:
# This column is what gets sent to the LLM for each row.
# We build it here so you can inspect it before making any API calls.

def build_prompt(row):
    hs_desc      = row.get("hs_desc")      or "Unknown goods"
    isic_descs   = row.get("isic4_descs")  or "Unknown"
    isic_sectors = row.get("isic4_sectors") or "Unknown"

    return f"""You are a customs trade compliance expert.

A company is registered for the following business activities:
- Registered activities: {isic_descs}
- Registered sector(s): {isic_sectors}

The company has submitted a trade declaration for the following goods:
- Goods description: {hs_desc}

Does the traded goods description logically match the company's registered business activities?

Respond in JSON only with exactly these three fields:
{{
  "mismatch": true or false,
  "risk_level": "normal" or "low" or "medium" or "high",
  "reason": "one sentence explanation"
}}

Rules:
- If the goods clearly fit the registered activities: mismatch false, risk_level normal
- If the goods are loosely related to the registered activities: mismatch false, risk_level normal
- If the goods are debatable to the registered activities: mismatch true, risk_level low
- If the goods are clearly unrelated to the registered activities: mismatch true, risk_level medium
- If the goods are completely unrelated AND suspicious (e.g. weapons, chemicals, controlled goods): mismatch true, risk_level high
- Do not output anything outside the JSON block."""


df["llm_prompt"] = df.apply(build_prompt, axis=1)


## Step 6 — LLM Caller Functions

In [7]:
# GPT-4o client
gpt_client = OpenAI(api_key=OPENAI_API_KEY)

def call_gpt4o(prompt: str) -> dict:
    try:
        response = gpt_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=150,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception as e:
        return {"mismatch": None, "risk_level": "error", "reason": str(e)}


# DeepSeek client
deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

def call_deepseek(prompt: str) -> dict:
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=150,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception as e:
        return {"mismatch": None, "risk_level": "error", "reason": str(e)}


def run_both_llms(prompt: str) -> dict:
    """Calls GPT-4o and DeepSeek and returns a combined result dict."""
    gpt  = call_gpt4o(prompt)
    deep = call_deepseek(prompt)
    return {
        "gpt4o_mismatch":    gpt.get("mismatch"),
        "gpt4o_risk":        gpt.get("risk_level"),
        "gpt4o_reason":      gpt.get("reason"),
        "deepseek_mismatch": deep.get("mismatch"),
        "deepseek_risk":     deep.get("risk_level"),
        "deepseek_reason":   deep.get("reason"),
    }


print("LLM caller functions ready.")

LLM caller functions ready.


## Step 7A — Run on All Rows

In [8]:
# The .head(100) cap is test on 100 rows first to verify outputs.

TARGET = df.copy().head(25)

results_all = []
for i, (idx, row) in enumerate(TARGET.iterrows()):
    if i % 10 == 0:
        print(f"Processing row {i} of {len(TARGET)}...")
    res = run_both_llms(row["llm_prompt"])
    results_all.append(res)
    time.sleep(0.3)

results_all_df   = pd.DataFrame(results_all, index=TARGET.index)
df_all_output    = pd.concat([TARGET, results_all_df], axis=1)

df_all_output.to_csv(OUTPUT_ALL, index=False)
print(f"Saved to: {OUTPUT_ALL}")
print(f"Shape:    {df_all_output.shape}")

Processing row 0 of 25...
Processing row 10 of 25...
Processing row 20 of 25...
Saved to: C:\Users\fawaz\Desktop\Project\outputs\CR_LLM.csv
Shape:    (25, 62)


## Step 8 — Summary of LLM Results

In [9]:
# Run this cell to see the risk distribution.
# Change df_all_output to df_flagged_output if you ran Step 7B.

OUTPUT_TO_SUMMARIZE = df_all_output

for model, risk_col, mismatch_col in [
    ("GPT-4o",   "gpt4o_risk",    "gpt4o_mismatch"),
    ("DeepSeek", "deepseek_risk", "deepseek_mismatch"),
]:
    print(f"\n{model} Risk Distribution:")
    if risk_col in OUTPUT_TO_SUMMARIZE.columns:
        print(OUTPUT_TO_SUMMARIZE[risk_col].value_counts(dropna=False))

    print(f"\n{model} Mismatch Count:")
    if mismatch_col in OUTPUT_TO_SUMMARIZE.columns:
        print(OUTPUT_TO_SUMMARIZE[mismatch_col].value_counts(dropna=False))


GPT-4o Risk Distribution:
gpt4o_risk
medium    25
Name: count, dtype: int64

GPT-4o Mismatch Count:
gpt4o_mismatch
True    25
Name: count, dtype: int64

DeepSeek Risk Distribution:
deepseek_risk
error    25
Name: count, dtype: int64

DeepSeek Mismatch Count:
deepseek_mismatch
None    25
Name: count, dtype: int64
